# 27. Ders: Yılın Şiiri'ni Web'e Taşımak (Gelişmiş Gradio Parametreleri)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Bir önceki derste tek bir girdi (kullanıcı metni) alıp analiz ettik. Ancak profesyonel yapay zeka ürünlerinde (Örneğin ChatGPT) kullanıcıya ayarlar sunulur: "Ne kadar uzun yazayım? Hangi dilde yazayım? Ne kadar yaratıcı olayım?"

Eğer kullanıcıya bu kontrol araçlarını (kaydırma çubukları, açılır menüler, onay kutuları) vermezsek, programımız çok kısıtlayıcı olur. Bu dersimizde 'Yılın Şiiri' (Year in Poetry) metin üretimi (Text Generation) projemizi Gradio web arayüzüne taşırken, arayüze birden fazla girdi bileşeni (Multiple Inputs) eklemeyi öğreneceğiz.

## Adım 2: Teknik Terimler ve Sözlük

- Multiple Inputs (Çoklu Girdi): Arayüzde kullanıcının dolduracağı birden fazla kutu veya bileşenin bir arada olması durumu.
- Slider (Kaydırma Çubuğu): Kullanıcının sayısal bir değeri (Örn: Temperature/Yaratıcılık skoru) minimum ve maksimum arasında kaydırarak seçtiği bileşen.
- Dropdown (Açılır Menü): Kullanıcının, kodcunun belirlediği seçenekler (Örn: Şiir Tarzı, Masal Tarzı) arasından tıklayarak birini seçtiği liste bileşeni.

## Adım 3: GPT-2 Modelini Yükleme

Önce yazarımızı (modelimizi) içeri alalım.

In [ ]:
import gradio as gr
from transformers import pipeline
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

yazar_model = pipeline("text-generation", model="gpt2")
print("GPT-2 Yazarlık motoru çalışmaya hazır.")

## Adım 4: Gelişmiş Prompt (İstem) Fonksiyonunu Yazma

Kullanıcı web sitesine girecek, 3 tane ilham kelimesi yazacak ve kaydırma çubuğu ile modelin ne kadar çılgın/mantıklı (Temperature) olacağını ayarlayacak. Yani fonksiyonumuz tam 4 tane parametre (girdi) alacak.

In [ ]:
def siir_makinesi(kelime1, kelime2, kelime3, yaraticilik_seviyesi):
    # Kullanıcının verdiği kelimelerle gelişmiş İngilizce (GPT-2 için) bir prompt yazıyoruz
    prompt = f"""
You are an award-winning poet.
Write a beautiful and inspiring poem using the following words: {kelime1}, {kelime2}, {kelime3}.
Poem:
"""
    
    # Modeli çağırıyoruz. Kullanıcının arayüzden seçtiği (Slider) sıcaklık değerini 'temperature' içine atıyoruz
    sonuc = yazar_model(
        prompt, 
        max_new_tokens=40, 
        temperature=yaraticilik_seviyesi, # Burası Slider'dan gelecek
        pad_token_id=50256
    )
    
    ham_ciktisi = sonuc[0]['generated_text']
    
    # Prompt kısmını kesip atıp sadece yazılan şiiri alma işlemi
    try:
        temiz_siir = ham_ciktisi.split("Poem:")[1].strip()
    except:
        # Eğer model beklenen formata uymazsa tüm çıktıyı ver
        temiz_siir = ham_ciktisi
        
    return temiz_siir

## Adım 5: gr.Interface'e Çoklu Girdi (Multiple Inputs) Verme

Gradio'da birden fazla girdi almak istiyorsak, `inputs` kısmına bir LİSTE (`[]`) tanımlamalıyız. Sıralama çok önemlidir: Listedeki 1. bileşen fonksiyonun 1. parametresine (`kelime1`), 4. bileşen ise 4. parametresine (`yaraticilik_seviyesi`) gider.

In [ ]:
# Bileşenleri tanımlayalım
k1 = gr.Textbox(label="İlham Kelimesi 1 (İngilizce Örn: Ocean)")
k2 = gr.Textbox(label="İlham Kelimesi 2 (İngilizce Örn: Star)")
k3 = gr.Textbox(label="İlham Kelimesi 3 (İngilizce Örn: Dream)")

# Yaratıcılık için Slider (Minimum 0.1, Maksimum 2.0, Varsayılan değer 0.7)
slider = gr.Slider(minimum=0.1, maximum=2.0, value=0.7, step=0.1, label="Yaratıcılık (Temperature)")

# Çıktı kutusunu büyük yapıyoruz (lines=10) çünkü şiir uzun olabilir
cikti = gr.Textbox(lines=10, label="🏆 Yılın Şiiri Orijinal Eseri")

# Arayüzü bağlıyoruz
siir_arayuzu = gr.Interface(
    fn=siir_makinesi,
    inputs=[k1, k2, k3, slider], # LİSTE halinde sırayla verdik
    outputs=cikti,
    title="🎭 Yapay Zeka Şair - Yılın Şiiri Üreticisi",
    description="3 adet İngilizce kelime girin. Slider ile yaratıcılık (delilik) seviyesini belirleyin ve yapay zekanın size özel bir şiir yazmasını izleyin."
)
print("Arayüz bileşenleri birbirine bağlandı.")

## Adım 6: Uygulamayı Başlatma

Projemizi internete açalım.

In [ ]:
siir_arayuzu.launch(share=True)

## Adım 7: Dene ve Keşfet - Farklı Ayarları Test Etmek

Uygulamaya girin. İlk denemede kelimeleri yazıp, Slider'ı (Yaratıcılığı) 0.1 (En düşük) yapıp butona basın. Sonucu inceleyin. Sonra hiçbir kelimeyi DEĞİŞTİRMEDEN Slider'ı 2.0 (En yüksek) yapın ve tekrar basın. Arayüzün gücü sayesinde koda dokunmadan modelin nasıl halüsinasyon gördüğüne veya tamamen uydurduğuna şahit olun.

## Adım 8: Sıra Sende Görevi - Şaka Şefi (Dropdown Kullanımı)

Problem: Kullanıcıların seçtiği bir konuya göre onlara bir fıkra (veya espri) yazan eğlenceli bir bot arayüzü tasarlamanız isteniyor. Ancak kullanıcı konuyu kendi yazmayacak, sizin verdiğiniz bir listeden seçecek.

Gereksinimler:
1. `saka_yap(konu)` adında bir fonksiyon oluşturun.
2. Fonksiyonda promptunuz şu şekilde olsun: `f"Tell a short, funny joke about {konu}:"`
3. Modeli çalıştırıp çıkan fıkrayı (`temiz_ciktisi`) return edin.
4. Arayüz kısmında `inputs` olarak bir Dropdown (Açılır Menü) ekleyin.
   *Kullanım Şekli:* `inputs=gr.Dropdown(choices=["Cats", "Programmers", "Space", "School"], label="Şaka Konusu Seçin")`
5. Çıktı olarak `gr.Textbox` kullanın ve uygulamanızı `launch()` ile yayınlayıp test edin.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 27. Dersi tamamlayarak Text Generation (Metin Üretim) modellerini ileri düzey bileşenler (Slider, Dropdown) ile kullanıcıların hizmetine sunmayı başardınız. Bu, modern SaaS (Hizmet Olarak Yazılım) ürünlerinin arka planında çalışan temel mekanizmadır.

Bir sonraki dersimizde, metin benzerliği arayan (TF-IDF ve Cosine Similarity) Buzdolabı Şefi (Fridge Chef) algoritmamızı, içinde HTML kodları ile şık görünüme sahip bir web menüsüne taşıyacağız.